# Part 1: Load Data

In [1]:
import pandas as pd
from nlp4bia.datasets.benchmark.medprocner import MedprocnerLoader, MedprocnerGazetteer
from nlp4bia.linking.retrievers import DenseRetriever
from sentence_transformers import SentenceTransformer

/gpfs/projects/bsc14/code/nlp4bia/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "/gpfs/projects/bsc14/abecerr1/hub/models--ICB-UMA--ClinLinker-KB-GP/snapshots/8f914c58a1cbcff43331eb15b101eaa5e5c6920a"

st_model = SentenceTransformer(model_name)

df_proc = MedprocnerLoader().df
gaz_proc = MedprocnerGazetteer().df

No sentence-transformers model found with name /gpfs/projects/bsc14/abecerr1/hub/models--ICB-UMA--ClinLinker-KB-GP/snapshots/8f914c58a1cbcff43331eb15b101eaa5e5c6920a. Creating a new one with mean pooling.


In [3]:
ls_terms = gaz_proc["term"].tolist()
ls_mentions = df_proc["span"].tolist()[:10]
term2code = gaz_proc.set_index("term")["code"].to_dict()

print("Gazetteer:\t\t", ls_terms[:5])
print("Dataset:\t\t", ls_mentions[:5])
print("Gazetteer dict format:", list(term2code.items())[:5])

Gazetteer:		 ['útero en el posparto: disminución del sangrado', 'última visita del médico al paciente internado con instrucciones en el momento del alta', 'úlcera isquémica ausente', 'óxidos de plomo', 'óxidos']
Dataset:		 ['Auscultación pulmonar', 'exploración neurológica', 'exploración urológica', 'palpación', 'transiluminación']
Gazetteer dict format: [('útero en el posparto: disminución del sangrado', '386219007'), ('última visita del médico al paciente internado con instrucciones en el momento del alta', '83362003'), ('úlcera isquémica ausente', '733740006'), ('óxidos de plomo', '311745006'), ('óxidos', '272156001')]


# Part 2: Do the Dense Retrieval

In [4]:
# Dense Retriever step 0: Build vector database
vector_db = st_model.encode(ls_terms, 
                            show_progress_bar=True, 
                            convert_to_tensor=True,
                            batch_size=4096)

biencoder = DenseRetriever(df_candidates=gaz_proc, vector_db=vector_db, model_or_path=st_model)

Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 58/58 [00:26<00:00,  2.16it/s]


In [5]:
ls_medprocner_train = biencoder.retrieve_top_k(
                                                ls_mentions, 
                                                k=200, 
                                                input_format="text",
                                                return_documents=True
                                            )

len(ls_medprocner_train)

Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 87.14it/s]


10

In [6]:
ls_medprocner_train[0].keys()

dict_keys(['codes', 'terms', 'similarity', 'mention'])

In [7]:
print(ls_medprocner_train[1]["codes"][:5])
print(ls_medprocner_train[1]["terms"][-5:])
print(ls_medprocner_train[1]["similarity"][:5])
print(ls_medprocner_train[1]["mention"])

['225398001', '84728005', '225398001', '372057006', '170692007']
['examen de muslo', 'exploración del ojo', 'evaluación de función motriz fina', 'evaluación del comportamiento basal', 'procedimiento de evaluación']
[0.9793567061424255, 0.9693054556846619, 0.8901357054710388, 0.8082420229911804, 0.7928519248962402]
exploración neurológica


# Part 3: Do the re-ranking of the top k retrieved candidates

In [8]:
from nlp4bia.linking.rerankers import CrossEncoderReranker

# Example:
ce_model_path = "/gpfs/projects/bsc14/MN4/bsc14/models/entity_linking/procedimiento/crossencoder_medprocner_5_epoch_16_batch"
reranker = CrossEncoderReranker(
                                    model_or_path=ce_model_path,
                                    device="cuda",
                                    batch_size=4096,
                                    term2code=term2code,
                                    show_progress_bar=True,
                                )

# Run reranking:
ls_reranked = reranker.rerank(ls_mentions[:10], ls_medprocner_train[:10], return_documents=True, k=10)

Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.10it/s]


In [9]:
ls_reranked[0].keys()

dict_keys(['terms', 'codes', 'similarity', 'mention'])

In [10]:
print(ls_reranked[2]["codes"][:5])
print(ls_reranked[2]["terms"][:5])
print(ls_reranked[2]["similarity"][:5])
print(ls_reranked[2]["mention"])

['302778005', '268945009', '363117004', '163351008', '281011001']
['examen urológico', 'examen del aparato genitourinario', 'exploración del aparato genitourinario', 'examen genitourinario completo', 'examen de las vías urinarias']
[0.9717382192611694, 0.9393976926803589, 0.8606930375099182, 0.018942806869745255, 0.006710724905133247]
exploración urológica
